# XGBM Model Training
This document contains the code for training the XGBM model to predict fraud from the transaction dataset.

## Set Up




In [ ]:
# run once per environment
# pip install lightgbm

In [ ]:
# install libaries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import lightgbm as lgb
import xgboost as xgb
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

In [ ]:
# load data
X_train = pd.read_csv("/content/train_transaction.csv")
X_val = pd.read_csv("/content/val_transaction.csv")

In [ ]:
# split data sets
y_train = X_train['isFraud']
X_train.drop(['isFraud'], axis=1, inplace=True)

y_val = X_val['isFraud']
X_val.drop(['isFraud'], axis=1, inplace=True)

In [ ]:
X_train.head()
X_train.drop(['Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0'], axis=1, inplace=True)
X_val.drop(['Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0'], axis=1, inplace=True)

### Can we drop any email domains to reduce dimensionality?

In [ ]:
# convert to light gbm datasets
train_data = lgb.Dataset(X_train, label=y_train)

In [ ]:
# create model (thanks for the params, katie!)
lgb_params = {'num_leaves': 256,
          'min_child_samples': 79,
          'objective': 'binary',
          'max_depth': 13,
          'learning_rate': 0.03,
          'boosting_type': 'gbdt',
          'subsample_freq': 3,
          'subsample': 0.9,
          'bagging_seed': 11,
          'metric': 'auc',
          "verbosity": -1,
          'reg_alpha': 0.3,
          'reg_lambda': 0.3,
          'colsample_bytree': 0.9
         }

result_lgb = lgb.train(lgb_params, train_data, num_boost_round=100)

In [ ]:
# get predicted results
y_pred_lgb = result_lgb.predict(X_val)

In [ ]:
# get accuracy
accuracy_lgb = accuracy_score(y_val, (y_pred_lgb > 0.5).astype(int))
roc_auc_lgb = roc_auc_score(y_val, y_pred_lgb)

print(f'Accuracy: {accuracy_lgb}')
print(f'ROC AUC: {roc_auc_lgb}')

In [ ]:
# view importance
lgb.plot_importance(result_lgb, height=0.2, title='Feature importance', xlabel='Feature importance', ylabel='Features', importance_type='auto', grid=True, max_num_features=30, precision=3)

No emails cracked the top 30, so I feel safe dropping it.

In [ ]:
X_train_v2 = X_train[X_train.columns.drop(list(X_train.filter(regex='email')))]
X_val_v2 = X_val[X_val.columns.drop(list(X_val.filter(regex='email')))]

## XGBoost Model

Now that we used LightGBM to see if it was okay to drop the email domains, let's make the XGBoost model

In [ ]:
# create validation and train data sets
train_data = xgb.DMatrix(X_train_v2, label=y_train, enable_categorical=True)
val_data = xgb.DMatrix(X_val_v2, label=y_val, enable_categorical=True)

In [ ]:
# train model
xgb_params = {
    'max_leaves': 256,
    'min_child_weight': 79,
    'objective': 'binary:logistic',
    'max_depth': 13,
    'eta': 0.03,
    'tree_method': 'hist',
    'subsample': 0.9,
    'colsample_bytree': 0.9,
    'seed': 11,
    'eval_metric': 'auc',
    'verbosity': 0,
    'reg_alpha': 0.3,
    'reg_lambda': 0.3,
    'grow_policy': 'lossguide'  # needed so max_leaves is actually respected
}

result_xgb = xgb.train(
    xgb_params,
    train_data,
    num_boost_round=100,
    evals=[(train_data, 'train'), (val_data, 'valid')],
    verbose_eval=10
)

In [ ]:
y_pred_xgb = result_xgb.predict(val_data)

In [ ]:
accuracy_xgb = accuracy_score(y_val, (y_pred_xgb > 0.5).astype(int))
roc_auc_xgb = roc_auc_score(y_val, y_pred_xgb)

print(f'Accuracy: {accuracy_xgb}')
print(f'ROC AUC: {roc_auc_xgb}')

In [ ]:
xgb.plot_importance(result_xgb, height=0.2, title='Feature importance', xlabel='Feature importance', ylabel='Features', max_num_features=30, grid=True)

### XGBoost with Emails

In [ ]:
# create validation and train data sets
train_v2 = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
val_v2 = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)

In [ ]:
# train model
result_xgb = xgb.train(
    xgb_params,
    train_v2,
    num_boost_round=100,
    evals=[(train_v2, 'train'), (val_v2, 'valid')],
    verbose_eval=10
)

In [ ]:
y_pred_xgb = result_xgb.predict(val_v2)

In [ ]:
accuracy_xgb = accuracy_score(y_val, (y_pred_xgb > 0.5).astype(int))
roc_auc_xgb = roc_auc_score(y_val, y_pred_xgb)

print(f'Accuracy: {accuracy_xgb}')
print(f'ROC AUC: {roc_auc_xgb}')

In [ ]:
xgb.plot_importance(result_xgb, height=0.2, title='Feature importance', xlabel='Feature importance', ylabel='Features', max_num_features=30, grid=True)

The ROC-AUC is _slightly_ better than the model w/o the emails, but not by much. The actually accuracy is the same.

It appears that the only email domain worth keeping in would be P_emaildomain_gmail.com, which we can examine to see how its correlated with isFraud.

### Examining P_emaildomain_gmail.com

In [ ]:
corr_matrix = X_train[['P_emaildomain_gmail.com', 'isFraud']].corr()
corr_matrix

P_emaildomain_gmail.com is not highly correlated with isFraud, so I'm curious about what its combining with that is making it crack the top 30!